In [ ]:
# ================================================================ half-wave plates
# Run this cell first, then the simulator cell below. Re-run both after changing anything here.
# Angles are fast-axis angles in degrees from H (the table plane).
#   HWP0 input, before PBS1          HWP3 arm C, between PBS3 and M2
#   HWP1 between PBS1 and PBS3       HWP4 between PBS2 and PBS4
#   HWP2 arm A, between M1 and PBS2  HWP5 between PBS4 and the analyzer PBS
import importlib
import interferometer_sim
importlib.reload(interferometer_sim)
from interferometer_sim import equal_intensity_angles, jones_paths, HWP_NAMES, HWP_WHERE
import numpy as np

LASER_POL_DEG = 0.0   # laser polarization angle from H
PBS_LEAK = 0.0        # PBS leakage, e.g. 1e-3 for a 1000:1 cube

HWP_DEG = equal_intensity_angles(LASER_POL_DEG)   # equal intensity from A, B and C
# Override individual plates here, e.g.
# HWP_DEG["HWP5"] = 0.0

for n in HWP_NAMES:
    print(f"{n}  {HWP_DEG[n]:7.2f}°   {HWP_WHERE[n]}")
J = jones_paths(HWP_DEG, LASER_POL_DEG, PBS_LEAK)
P = {p: float(np.sum(np.abs(v)**2)) for p, v in J.items()}
print("\npower to camera (fraction of laser):", "  ".join(f"{p}: {100*P[p]:.2f}%" for p in P),
      f"  total {100*sum(P.values()):.1f}%")


# Three-path interferometer – live fringe simulator

Interactive model of the camera pattern from the three-path PBS interferometer (λ₀ = 810 nm).
The three beams reach the camera **side by side** along x (A left, B middle, C right) and slightly tilted toward each other, so where they overlap they form straight fringes.

| path | launching optic (stage + tilt) | phase shifter |
|---|---|---|
| A | PBS1 (reflected arm) | slide θ_A |
| B | PBS3 (reflected arm) | slide θ_B |
| C | M2 (right-hand mirror) | – (reference arm) |

The simulated camera is the PI-MAX 4 sensor: **1024 × 1024 pixels** (12.8 µm pixels ≈ 13.1 mm across). `VIEW_PX` sets whether the full sensor or a centred 512 / 256 px window is shown; fringe periods are reported in pixels.

**Polarization** is modelled with Jones matrices: each HWP rotates the polarization by twice its angle, each PBS transmits H and reflects V (optional leakage). The light each path brings to the camera therefore depends on the six HWP angles. The angles are set in the **half-wave plate cell at the top**. By default it uses the equal-intensity angles HWP0 = 17.63°, HWP1 = 22.5°, HWP2 = HWP3 = 45°, HWP4 = 22.5°, HWP5 = 17.63° (for the laser polarization `LASER_POL_DEG`, default H), giving 1/9 of the laser power from each path, the most possible with equal intensities. Brightness is normalised to that ideal, so a mis-set plate shows up as a dimmer or lopsided pattern.

**Setup parameters** (wavelength, slide thickness, beam waist, pixel size, view size, lever arms) are constants at the top of the simulator cell; laser polarization, PBS leakage and HWP angles are in the half-wave plate cell. Edit them and re-run the cells (half-wave plate cell first).

**What each control does**
- **Slide θ** – tilting a microscope slide changes that path's phase (and shifts the beam slightly sideways).
- **Stage** – translating PBS1 / PBS3 / M2 along the incoming beam moves that beam sideways on the camera *and* lengthens the path by the same amount. At 810 nm, 1 µm of stage is ≈ 7.8 rad, so stages are for overlap; set phases with the slides. Set `STAGE_CHANGES_PHASE = False` to see only the position effect.
- **Tilt x / y** – tilting the optic by δ steers the beam by 2δ: this sets the fringe spacing and orientation, and moves the beam by 2δ·L on the camera (L = lever arm to the camera, set in `Setup.lever_mm`).

The physics lives in `interferometer_sim.py` (`Setup`), so the same model can be used in analysis scripts. Assumptions: Gaussian beams, ideal mirrors, BK7-like slide index.


In [ ]:
import io
import numpy as np
import ipywidgets as W
from IPython.display import display
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

import importlib
import interferometer_sim
importlib.reload(interferometer_sim)  # pick up edits to the module without restarting the kernel
from interferometer_sim import Setup, PATHS, LAUNCH_OPTIC, slide_slope, slide_walk, draw_setup

assert "HWP_DEG" in globals(), "Run the half-wave plate cell at the top of the notebook first."

# ================================================================ setup (edit here)
LAMBDA_NM = 810.0            # laser wavelength
SLIDE_T_MM = 1.0             # microscope-slide thickness
WAIST_MM = 1.0               # beam waist at the camera
STAGE_CHANGES_PHASE = True   # translating PBS1 / PBS3 / M2 also lengthens that path
PIXEL_UM = 12.8              # PI-MAX 4 pixel pitch (1024 x 1024 sensor)
VIEW_PX = 512                # camera window shown: 1024 (full sensor), 512 or 256
LEVER_MM = {"A": 600.0, "B": 450.0, "C": 300.0}   # launch optic -> camera distance


def make_setup():
    # HWP_DEG, LASER_POL_DEG and PBS_LEAK come from the half-wave plate cell at the top
    return Setup(lam_nm=LAMBDA_NM, input_pol_deg=LASER_POL_DEG, slide_t_mm=SLIDE_T_MM,
                 waist_mm=WAIST_MM, pbs_leak=PBS_LEAK, stage_changes_phase=STAGE_CHANGES_PHASE,
                 pixel_um=PIXEL_UM, lever_mm=dict(LEVER_MM), hwp_deg=dict(HWP_DEG))
# ================================================================================


setup = make_setup()
COLORS = {"A": "#4fc3f7", "B": "#ffd54f", "C": "#ef5350"}

# ---------------------------------------------------------------- widgets
def fslider(desc, v, lo, hi, step, fmt):
    return W.FloatSlider(value=v, min=lo, max=hi, step=step, description=desc,
                         readout_format=fmt, continuous_update=True,
                         style={"description_width": "110px"}, layout=W.Layout(width="420px"))

thA = fslider("slide θ_A (°)", setup.theta_A_deg, 0, 30, 0.001, ".3f")
thB = fslider("slide θ_B (°)", setup.theta_B_deg, 0, 30, 0.001, ".3f")

stage, tx, ty, blk = {}, {}, {}, {}
tabs = []
for p in PATHS:
    L = setup.launch[p]
    stage[p] = fslider(f"{LAUNCH_OPTIC[p]} stage (mm)", L.stage_mm, -1.5, 1.5, 0.0005, ".4f")
    tx[p] = fslider(f"{LAUNCH_OPTIC[p]} tilt x (mrad)", L.tilt_x_mrad, -2, 2, 0.005, ".3f")
    ty[p] = fslider(f"{LAUNCH_OPTIC[p]} tilt y (mrad)", L.tilt_y_mrad, -2, 2, 0.005, ".3f")
    blk[p] = W.Checkbox(value=L.blocked, description=f"block path {p}")
    tabs.append(W.VBox([stage[p], tx[p], ty[p], blk[p]]))
beam_tabs = W.Tab(children=tabs)
for i, p in enumerate(PATHS):
    beam_tabs.set_title(i, f"{p}  ({LAUNCH_OPTIC[p]})")

info = W.HTML()
img = W.Image(format="png")

# ---------------------------------------------------------------- figure
fig = Figure(figsize=(11, 9.6), dpi=90)
FigureCanvasAgg(fig)
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.35], height_ratios=[1.15, 1], wspace=0.25, hspace=0.12,
                      left=0.06, right=0.98, top=0.98, bottom=0.06)
ax_set = fig.add_subplot(gs[0, :])
ax_cam = fig.add_subplot(gs[1, 0])
ax_pro = fig.add_subplot(gs[1, 1])
grid = {}


def make_grid():
    """Pixel window currently shown, and the mm coordinates of every pixel in it."""
    px = setup.pixel_window(VIEW_PX)
    c = setup.px_to_mm(px)
    X, Y = np.meshgrid(c, c)
    grid.update(px=px, c=c, X=X, Y=Y, row=len(px) // 2,
                extent=[px[0] - 0.5, px[-1] + 0.5, px[0] - 0.5, px[-1] + 0.5])


make_grid()
im = ax_cam.imshow(np.zeros((len(grid["px"]),) * 2), extent=grid["extent"], origin="lower",
                   cmap="inferno", vmin=0, vmax=1, interpolation="nearest")
ax_cam.set_title("PI-MAX 4 (simulated)")
ax_cam.set_xlabel("x (pixel)"); ax_cam.set_ylabel("y (pixel)")
hline = ax_cam.axhline(grid["px"][grid["row"]], color="w", lw=0.6, ls=":", alpha=0.6)
line_now, = ax_pro.plot(grid["px"], 0 * grid["c"], color="#5b45e0", lw=2, label="now")
env_lines = {p: ax_pro.plot(grid["px"], 0 * grid["c"], color=COLORS[p], lw=1, alpha=0.7, label=f"|E_{p}|²")[0] for p in PATHS}
ax_pro.set_ylim(0, 1.05)
ax_pro.set_xlabel("x (pixel)"); ax_pro.set_ylabel("I / I_max")
ax_pro.set_title("cross-section through the centre row")
ax_pro.grid(alpha=0.3); ax_pro.legend(loc="upper right", fontsize=8)


def read_widgets():
    setup.theta_A_deg, setup.theta_B_deg = thA.value, thB.value
    for p in PATHS:
        L = setup.launch[p]
        L.stage_mm, L.tilt_x_mrad, L.tilt_y_mrad, L.blocked = stage[p].value, tx[p].value, ty[p].value, blk[p].value


def redraw(*_):
    read_widgets()
    X, Y, px, c, row = grid["X"], grid["Y"], grid["px"], grid["c"], grid["row"]
    draw_setup(ax_set, setup)
    Imax = setup.max_intensity(X, Y)
    I = setup.intensity(X, Y) / Imax
    im.set_data(I); im.set_extent(grid["extent"])
    ax_cam.set_xlim(grid["extent"][:2]); ax_cam.set_ylim(grid["extent"][2:])
    hline.set_ydata([px[row], px[row]])
    prof = I[row]
    line_now.set_data(px, prof)
    ax_pro.set_xlim(px[0], px[-1])
    ax_pro.set_ylim(0, max(1.05, 1.05 * float(prof.max())))
    for p in PATHS:
        b = setup.beam(p)
        on = not b["blocked"]
        env = b["amp"]**2 * np.exp(-2 * ((c - b["x0"])**2 + (c[row] - b["y0"])**2) / setup.waist_mm**2) / Imax
        env_lines[p].set_data(px, env); env_lines[p].set_visible(on)
    buf = io.BytesIO(); fig.savefig(buf, format="png"); img.value = buf.getvalue()

    ph = setup.phases()
    fp, fpx = setup.fringe_periods(), setup.fringe_periods_px()
    fp_txt = ", ".join(f"{k}: {fpx[k]:.1f} px ({v:.3f} mm)" for k, v in fp.items()) or "–"
    sensor_mm = setup.npix * setup.pixel_um * 1e-3
    ang = ", ".join(f"{p}: {setup.beam(p)['angle_x_mrad']:+.2f}" for p in PATHS)
    pw = setup.camera_powers()
    pw_txt = " &nbsp; ".join(f"{p}: {100*pw[p]:.2f}%" for p in PATHS)
    spread = (max(pw.values()) - min(pw.values())) / max(max(pw.values()), 1e-12)
    pw_flag = "equal" if spread < 0.01 else f"<span style='color:#c0392b'>unequal (spread {100*spread:.0f}%)</span>"
    info.value = (
        "<div style='font-family:monospace;font-size:13px;line-height:1.6'>"
        f"power to camera (of laser): {pw_txt} &nbsp; total {100*sum(pw.values()):.1f}% &nbsp; {pw_flag}<br>"
        f"φ_A = {ph['A']:.3f} rad &nbsp; φ_B = {ph['B']:.3f} rad &nbsp; φ_C = {ph['C']:.3f} rad (mod 2π)<br>"
        f"slide dφ/dθ: A {slide_slope(setup.theta_A_deg, setup.lam_nm, setup.slide_t_mm):.1f}, "
        f"B {slide_slope(setup.theta_B_deg, setup.lam_nm, setup.slide_t_mm):.1f} rad/° &nbsp; "
        f"slide beam walk: A {1e3*slide_walk(setup.theta_A_deg, setup.lam_nm, setup.slide_t_mm):.0f} µm, "
        f"B {1e3*slide_walk(setup.theta_B_deg, setup.lam_nm, setup.slide_t_mm):.0f} µm<br>"
        f"beam angles x (mrad): {ang}<br>"
        f"fringe periods: {fp_txt}<br>"
        f"sensor: {setup.npix} × {setup.npix} px, {setup.pixel_um:.1f} µm pixels = {sensor_mm:.2f} mm across; "
        f"beam waist = {setup.waist_mm / (setup.pixel_um * 1e-3):.0f} px<br>"
        f"stage phase: 1 µm of stage = {2*np.pi*1e3/setup.lam_nm:.2f} rad</div>"
    )


for w in [thA, thB] + [d[p] for d in (stage, tx, ty, blk) for p in PATHS]:
    w.observe(redraw, names="value")

controls = W.VBox([
    W.HTML("<b>Phase shifters</b>"), thA, thB,
    W.HTML("<b>Launch optics</b>"), beam_tabs,
])
redraw()
display(W.VBox([img, controls, info]))


In [ ]:
# ================================================================ phase compensation
# A 630 nm reference laser shares all three paths with the 810 nm beam (dichroic DM1 at the
# input, achromatic HWPs). The beam blocks are short-pass filters, so the reference always sees
# A, B and C. After the analyzer, dichroic DM2 sends the 630 nm light to a reference camera.
# Its fringe pattern holds the AC and BC fringes at different spatial frequencies, so a Fourier
# demodulation reads phi_A - phi_C and phi_B - phi_C separately (no sign ambiguity).
# The PC converts the reference phase drift into a path-length drift, works out the 810 nm phase
# error it causes (including the slides' different dispersion at 630 and 810 nm), and turns the
# motorized slides to cancel it. Uses the current slider settings from the simulator cell.
import matplotlib.pyplot as plt
from interferometer_sim import draw_setup, simulate_phase_lock, fringe_profile, PATHS

assert "setup" in globals(), "Run the simulator cell first."

# ---------------------------------------------------------------- lock settings (edit here)
REF_NM = 630.0                                # reference laser wavelength
REF_PIXEL_UM = 5.2                            # reference camera pixel pitch (1024 px wide)
REF_NOISE = 0.02                              # reference camera pixel noise (fraction of max)
MOTOR_STEP_DEG = 0.001                        # rotation-stage resolution for the slides
N_STEPS, DT_S = 300, 1.0                      # loop updates and time between them (s)
WALK_NM = 3.0                                 # random path-length drift per arm, nm per sqrt(s)
RATE_NM_S = {"A": 2.0, "B": -1.0, "C": 0.5}   # steady thermal drift per arm (nm/s)

res = simulate_phase_lock(setup, ref_nm=REF_NM, ref_pixel_um=REF_PIXEL_UM, n_steps=N_STEPS,
                          dt_s=DT_S, walk_nm=WALK_NM, rate_nm_s=RATE_NM_S, ref_noise=REF_NOISE,
                          motor_step_deg=MOTOR_STEP_DEG)
t, ef, el = res["t"], res["err_free"], res["err_lock"]
rms = lambda e: np.sqrt(np.mean(e**2, axis=0))
print(f"810 nm phase error over {t[-1]:.0f} s      phi_A-phi_C    phi_B-phi_C")
print(f"  no compensation (RMS)        {rms(ef)[0]:8.3f} rad   {rms(ef)[1]:8.3f} rad")
print(f"  locked (RMS)                 {rms(el)[0]:8.4f} rad   {rms(el)[1]:8.4f} rad")
print(f"  locked (max)                 {np.abs(el).max(0)[0]:8.4f} rad   {np.abs(el).max(0)[1]:8.4f} rad")
print(f"slide travel: A {np.ptp(res['theta_A']):.3f}°, B {np.ptp(res['theta_B']):.3f}°  "
      f"(motor step {MOTOR_STEP_DEG}°)")

# ---------------------------------------------------------------- schematic
fig1, ax = plt.subplots(figsize=(12, 6.4))
draw_setup(ax, res["locked"], lock=True)
ax.set_title("Setup with active phase compensation", fontsize=12)
fig1.tight_layout()
plt.show()

# ---------------------------------------------------------------- reference camera + lock performance
ref, lock = res["ref"], res["lock"]
fig2, axs = plt.subplots(3, 2, figsize=(12, 12))
px, Iref = ref.camera(512)
axs[0, 0].imshow(Iref / Iref.max(), extent=[px[0], px[-1], px[0], px[-1]], origin="lower",
                 cmap="inferno", interpolation="nearest")
axs[0, 0].set_title(f"reference camera ({REF_NM:.0f} nm, centre 512 px)")
axs[0, 0].set_xlabel("x (pixel)"); axs[0, 0].set_ylabel("y (pixel)")

x, prof = fringe_profile(ref)
n = 16 * len(prof)
F = np.abs(np.fft.rfft((prof - prof.mean()) * np.hanning(len(prof)), n))
k = 2 * np.pi * np.fft.rfftfreq(n, d=x[1] - x[0])
axs[0, 1].plot(k, F / F.max(), color="#5b45e0", lw=1.2)
kAB = abs(ref.beam("A")["kx"] - ref.beam("B")["kx"])
axs[0, 1].text(kAB, 1.02, "AB\n(not used)", ha="center", color="#888", fontsize=8)
for pair, kp in lock.k.items():
    axs[0, 1].axvline(abs(kp), color="#ff8c00", ls="--", lw=1)
    axs[0, 1].text(abs(kp), 1.02, pair, ha="center", color="#ff8c00", fontsize=10, fontweight="bold")
axs[0, 1].set_xlim(0, 1.6 * max(abs(v) for v in lock.k.values()))
axs[0, 1].set_ylim(0, 1.2)
axs[0, 1].set_xlabel("spatial frequency k (rad/mm)"); axs[0, 1].set_ylabel("|FFT| (norm.)")
axs[0, 1].set_title("reference fringe spectrum: phases read at the AC and BC peaks")

for p, c in zip(PATHS, ("#4fc3f7", "#ffd54f", "#ef5350")):
    axs[1, 0].plot(t, res["dL_" + p], color=c, lw=1.5, label=f"arm {p}")
axs[1, 0].set_xlabel("time (s)"); axs[1, 0].set_ylabel("path-length drift (nm)")
axs[1, 0].grid(alpha=0.3); axs[1, 0].legend(fontsize=8)
axs[1, 0].set_title("simulated thermal drift")

labs = [("φ_A − φ_C", "#4fc3f7"), ("φ_B − φ_C", "#ef5350")]
ef_unw = np.unwrap(ef, axis=0)
for i, (lab, c) in enumerate(labs):
    axs[1, 1].plot(t, ef_unw[:, i], color=c, lw=1.2, alpha=0.6, label=f"{lab}, no compensation")
    axs[1, 1].plot(t, el[:, i], color=c, lw=2.2, label=f"{lab}, locked")
axs[1, 1].set_xlabel("time (s)"); axs[1, 1].set_ylabel("810 nm phase error (rad)")
axs[1, 1].grid(alpha=0.3); axs[1, 1].legend(fontsize=8)
axs[1, 1].set_title("phase error at the PI-MAX")

for i, (lab, c) in enumerate(labs):
    axs[2, 0].plot(t, 1e3 * el[:, i], color=c, lw=1.2, label=lab)
axs[2, 0].set_xlabel("time (s)"); axs[2, 0].set_ylabel("residual error (mrad)")
axs[2, 0].grid(alpha=0.3); axs[2, 0].legend(fontsize=8)
axs[2, 0].set_title("locked: residual phase error (zoom)")

axs[2, 1].plot(t, res["theta_A"], color="#4fc3f7", lw=2, label="slide A")
axs[2, 1].plot(t, res["theta_B"], color="#ffd54f", lw=2, label="slide B")
axs[2, 1].set_xlabel("time (s)"); axs[2, 1].set_ylabel("slide angle (°)")
axs[2, 1].grid(alpha=0.3); axs[2, 1].legend(fontsize=8)
axs[2, 1].set_title("slide angles commanded by the lock")
fig2.tight_layout()
plt.show()

# ---------------------------------------------------------------- PI-MAX cross-section at the end
import copy
start = copy.deepcopy(res["free"])
for L in start.launch.values():
    L.drift_rad = 0.0
start.theta_A_deg, start.theta_B_deg = setup.theta_A_deg, setup.theta_B_deg
fig3, ax3 = plt.subplots(figsize=(12, 3.6))
pxm = start.pixel_window(VIEW_PX)
cm = start.px_to_mm(pxm)
Xm, Ym = np.meshgrid(cm, cm[len(cm) // 2: len(cm) // 2 + 1])
norm = start.max_intensity(*np.meshgrid(cm, cm))
for s_, lab, sty in [(start, "at lock (target)", dict(color="k", lw=2.5, alpha=0.35)),
                     (res["free"], f"after {t[-1]:.0f} s, no compensation", dict(color="#c0392b", lw=1.2)),
                     (res["locked"], f"after {t[-1]:.0f} s, locked", dict(color="#0d8774", lw=1.5, ls="--"))]:
    ax3.plot(pxm, s_.intensity(Xm, Ym)[0] / norm, label=lab, **sty)
ax3.set_xlim(pxm[0], pxm[-1]); ax3.set_xlabel("x (pixel)"); ax3.set_ylabel("I / I_max")
ax3.set_title("PI-MAX cross-section (810 nm)"); ax3.grid(alpha=0.3); ax3.legend(fontsize=8)
fig3.tight_layout()
plt.show()
